<a href="https://colab.research.google.com/github/Somrat390/EDA-Practise/blob/main/Personal_Email_Data_EDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install mailbox

  Preparing metadata (setup.py) ... done
  Created wheel for mailbox: filename=mailbox-0.4-py3-none-any.whl size=4683 sha256=1ad1c8edc815270bd99201a83d53fea620c5bd4aa0d2e79a8dc56b4dd12d079f
  Stored in directory: /root/.cache/pip/wheels/06/cd/9a/64b75da2511d797260d3b3cb8cfbf66e700119cc045a9be2c9
Successfully built mailbox


In [26]:
import numpy as np
import pandas as pd
import mailbox
import matplotlib.pyplot as plt

In [ ]:
mboxfile = "/content/All mail Including Spam and Trash.mbox"
mbox = mailbox.mbox(mboxfile)
mbox

In [ ]:
for key in mbox[0].keys():
  print(key)

X-GM-THRID
X-Gmail-Labels
Delivered-To
Received
X-Received
ARC-Seal
ARC-Message-Signature
ARC-Authentication-Results
Return-Path
Received
Received-SPF
Authentication-Results
DKIM-Signature
DKIM-Signature
Date
From
Message-ID
Subject
MIME-Version
Content-Type
To
X-LinkedIn-Class
X-LinkedIn-Template
X-LinkedIn-fbl
X-LinkedIn-Id
List-Unsubscribe
List-Unsubscribe-Post
Feedback-ID
Require-Recipient-Valid-Since


# Data Cleansing

In [ ]:
import csv

In [ ]:
from os import write
with open('mailbox.csv','w') as outputfile:
  writer = csv.writer(outputfile)
  writer.writerow(['subject','from','date','to','label','thread'])

  for message in mbox:
    writer.writerow([
        message['subject'],
        message['from'],
        message['date'],
        message['to'],
        message['X-Gmail-Labels'],
        message['X-GM-THRID']
    ])

In [63]:
dfs = pd.read_csv('mailbox.csv', names=['subject', 'from', 'date', 'to', 'label', 'thread'])

In [34]:
dfs.dtypes

,0
subject,object
from,object
date,object
to,object
label,object
thread,object


In [64]:
dfs['date'] = dfs['date'].apply(lambda x: pd.to_datetime(x, errors='coerce', utc=True))

In [65]:
dfs = dfs[dfs['date'].notna()]

In [68]:
dfs['date'].isnull().sum()

np.int64(0)

In [69]:
dfs.to_csv('gmail.csv')

# Applying Descriptive statistics

In [70]:
dfs.info()

<class 'pandas.core.frame.DataFrame'>
Index: 365 entries, 1 to 365
Data columns (total 6 columns):
 #   Column   Non-Null Count  Dtype              
---  ------   --------------  -----              
 0   subject  365 non-null    object             
 1   from     365 non-null    object             
 2   date     365 non-null    datetime64[ns, UTC]
 3   to       365 non-null    object             
 4   label    365 non-null    object             
 5   thread   365 non-null    object             
dtypes: datetime64[ns, UTC](1), object(5)
memory usage: 20.0+ KB


In [ ]:
dfs.head()

,subject,from,date,to,label,thread
1,BL310 + Node-RED + MQTT: Building a Lightweigh...,"""Beilai Tech. Co., Ltd. -- AI Edge IPC, ARM IP...",2026-03-26 10:01:39+00:00,MD SOMRAT SHEIKH <somratsheikh426@gmail.com>,"Inbox,Category updates,Unread",1860718358751918478
2,Your Certification journey starts now!,Kiran at DataCamp <team@datacamp.com>,2026-03-24 03:17:45+00:00,somratsheikh426@gmail.com,"Inbox,Category updates,Unread",1860511744323717489
3,Your Statement of Accomplishment has Arrived,The DataCamp Team <team@datacamp.com>,2026-03-24 19:14:16+00:00,somratsheikh426@gmail.com,"Inbox,Opened,Category updates",1860571923793458079
4,The trap most developers fall into with AI cod...,Mosh Hamedani <newsletter@codewithmosh.com>,2026-03-25 12:32:04+00:00,somratsheikh426@gmail.com,"Inbox,Category updates,Unread",1860642894901688530
5,Updates to our Privacy Policy,The DataCamp Team <team@reply.datacamp.com>,2026-03-21 09:16:19+00:00,somratsheikh426@gmail.com,"Inbox,Important,Opened,Category updates",1860262511917002242


# As we need only the email not the name from the from column so we will remove it|


In [71]:
import re

In [72]:
def extract_email_ID(string):
  email = re.findall(r'<(.+?)>', string)
  if not email:
    email = list(filter(lambda y: '@' in y, string.split()))
  return email[0] if email else np.nan

In [73]:
dfs['from'] = dfs['from'].apply(lambda x: extract_email_ID(x))


In [74]:
myemail = 'somratsheikh426@gmail.com'
dfs['label'] = dfs['from'].apply(lambda x: 'sent' if x==myemail else 'inbox')

In [75]:
dfs.head()

,subject,from,date,to,label,thread
1,BL310 + Node-RED + MQTT: Building a Lightweigh...,newsletters-noreply@linkedin.com,2026-03-26 10:01:39+00:00,MD SOMRAT SHEIKH <somratsheikh426@gmail.com>,inbox,1860718358751918478
2,Your Certification journey starts now!,team@datacamp.com,2026-03-24 03:17:45+00:00,somratsheikh426@gmail.com,inbox,1860511744323717489
3,Your Statement of Accomplishment has Arrived,team@datacamp.com,2026-03-24 19:14:16+00:00,somratsheikh426@gmail.com,inbox,1860571923793458079
4,The trap most developers fall into with AI cod...,newsletter@codewithmosh.com,2026-03-25 12:32:04+00:00,somratsheikh426@gmail.com,inbox,1860642894901688530
5,Updates to our Privacy Policy,team@reply.datacamp.com,2026-03-21 09:16:19+00:00,somratsheikh426@gmail.com,inbox,1860262511917002242


# Dropping column

To column only contains my email so we can drop this irrelevant column

In [76]:
dfs.drop(columns='to', inplace=True)

In [45]:
dfs.head()

,subject,from,date,label,thread
1,BL310 + Node-RED + MQTT: Building a Lightweigh...,newsletters-noreply@linkedin.com,2026-03-26 10:01:39+00:00,inbox,1860718358751918478
2,Your Certification journey starts now!,team@datacamp.com,2026-03-24 03:17:45+00:00,inbox,1860511744323717489
3,Your Statement of Accomplishment has Arrived,team@datacamp.com,2026-03-24 19:14:16+00:00,inbox,1860571923793458079
4,The trap most developers fall into with AI cod...,newsletter@codewithmosh.com,2026-03-25 12:32:04+00:00,inbox,1860642894901688530
5,Updates to our Privacy Policy,team@reply.datacamp.com,2026-03-21 09:16:19+00:00,inbox,1860262511917002242


In [46]:
dfs.info()

<class 'pandas.core.frame.DataFrame'>
Index: 365 entries, 1 to 365
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype              
---  ------   --------------  -----              
 0   subject  365 non-null    object             
 1   from     365 non-null    object             
 2   date     365 non-null    datetime64[ns, UTC]
 3   label    365 non-null    object             
 4   thread   365 non-null    object             
dtypes: datetime64[ns, UTC](1), object(4)
memory usage: 17.1+ KB


# Refactor Timezone

In [77]:
import datetime
import pytz
def refactor_timezone(x):
  est = pytz.timezone('US/Eastern')
  return x.astimezone(est)


In [78]:
dfs['date'] = dfs['date'].apply(lambda x: refactor_timezone(x))

In [79]:
dfs['date']

,date
1,2026-03-26 06:01:39-04:00
2,2026-03-23 23:17:45-04:00
3,2026-03-24 15:14:16-04:00
4,2026-03-25 08:32:04-04:00
5,2026-03-21 05:16:19-04:00
...,...
361,2026-01-20 13:09:34-05:00
362,2026-01-22 20:30:00-05:00
363,2026-02-06 03:58:18-05:00
364,2026-03-25 00:13:54-04:00


In [80]:
dfs['dayofweek'] = dfs['date'].apply(lambda x: x.day_name)


In [81]:
dfs['dayofweek']

,dayofweek
1,<bound method _Timestamp.day_name of Timestamp...
2,<bound method _Timestamp.day_name of Timestamp...
3,<bound method _Timestamp.day_name of Timestamp...
4,<bound method _Timestamp.day_name of Timestamp...
5,<bound method _Timestamp.day_name of Timestamp...
...,...
361,<bound method _Timestamp.day_name of Timestamp...
362,<bound method _Timestamp.day_name of Timestamp...
363,<bound method _Timestamp.day_name of Timestamp...
364,<bound method _Timestamp.day_name of Timestamp...


In [82]:
dfs['dayofweek'] = pd.Categorical(dfs['dayofweek'], categories=[
    'Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday',
    'Saturday', 'Sunday'], ordered=True)

In [84]:
dfs['dayofweek']

,dayofweek
1,NaN
2,NaN
3,NaN
4,NaN
5,NaN
...,...
361,NaN
362,NaN
363,NaN
364,NaN


In [85]:
dfs['timeofday'] = dfs['date'].apply(lambda x: x.hour + x.minute/60 + x.second/3600)

In [86]:
dfs['dayofweek']

,dayofweek
1,NaN
2,NaN
3,NaN
4,NaN
5,NaN
...,...
361,NaN
362,NaN
363,NaN
364,NaN


In [87]:
dfs['timeofday']

,timeofday
1,6.027500
2,23.295833
3,15.237778
4,8.534444
5,5.271944
...,...
361,13.159444
362,20.500000
363,3.971667
364,0.231667


In [88]:
dfs['hour'] = dfs['date'].apply(lambda x: x.hour)
dfs['year_int'] = dfs['date'].apply(lambda x: x.year)
dfs['year'] = dfs['date'].apply(lambda x: x.year + x.dayofyear/365.25)

In [89]:
dfs.index = dfs['date']
del dfs['date']

In [90]:
dfs.head()

,subject,from,label,thread,dayofweek,timeofday,hour,year_int,year
date,,,,,,,,,
2026-03-26 06:01:39-04:00,BL310 + Node-RED + MQTT: Building a Lightweigh...,newsletters-noreply@linkedin.com,inbox,1860718358751918478,NaN,6.027500,6,2026,2026.232717
2026-03-23 23:17:45-04:00,Your Certification journey starts now!,team@datacamp.com,inbox,1860511744323717489,NaN,23.295833,23,2026,2026.224504
2026-03-24 15:14:16-04:00,Your Statement of Accomplishment has Arrived,team@datacamp.com,inbox,1860571923793458079,NaN,15.237778,15,2026,2026.227242
2026-03-25 08:32:04-04:00,The trap most developers fall into with AI cod...,newsletter@codewithmosh.com,inbox,1860642894901688530,NaN,8.534444,8,2026,2026.229979
2026-03-21 05:16:19-04:00,Updates to our Privacy Policy,team@reply.datacamp.com,inbox,1860262511917002242,NaN,5.271944,5,2026,2026.219028


# Data Analysis

How many emails did I send during a given timeframe?

In [91]:
print(dfs.index.min().strftime('%a, %d %b %Y %I:%M %p'))
print(dfs.index.max().strftime('%a, %d %b %Y %I:%M %p'))

print(dfs['label'].value_counts())

Tue, 13 Jan 2026 12:28 PM
Thu, 26 Mar 2026 06:51 PM
label
inbox    365
Name: count, dtype: int64


In [92]:
dfs.to_csv('Gmail.csv')